# Bài 1 — Học máy với dữ liệu dạng bảng (Adult Income)

**Môn học:** Học máy (Machine Learning) — Học kỳ I, năm học 2026-2027  
**Khoa Khoa học và Kỹ thuật Máy tính, Trường Đại học Bách Khoa, ĐHQG-HCM**  
**GVHD:** *[điền tên giảng viên]*  
**Nhóm:** *[tên nhóm]* — *[Họ tên – MSSV – email]* (2-3 thành viên)  
**Mã nguồn GitHub:** *[điền link repo nếu có]*

> **Hướng dẫn chạy:** chọn *Runtime → Change runtime type → T4 GPU* (khuyến nghị), sau đó *Runtime → Run all*.
> Notebook tự cài thư viện, tự tải dữ liệu công khai về máy chủ Colab, **không** mount Google Drive.

## 0. Cài đặt môi trường & nạp module

Các module Python hỗ trợ (`modules/*.py`) được ghi trực tiếp ra máy chủ Colab bằng `%%writefile` (cùng nội dung với thư mục `modules/` trong bài nộp / GitHub), nên notebook chạy độc lập mà không cần Drive.

In [ ]:
!pip -q install seaborn scikit-learn
!mkdir -p modules reports/figures features data

In [ ]:
%%writefile modules/common.py
"""Tiện ích dùng chung cho cả 3 bài: seed, đánh giá, vẽ hình, lưu kết quả/đặc trưng."""
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

REPORT_DIR = "reports"
FIG_DIR = os.path.join(REPORT_DIR, "figures")
FEATURE_DIR = "features"


def ensure_dirs():
    for d in (REPORT_DIR, FIG_DIR, FEATURE_DIR, "data"):
        os.makedirs(d, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def evaluate(y_true, y_pred, average: str = "macro") -> dict:
    """Accuracy, precision, recall, F1 (average = 'macro' | 'weighted' | 'binary')."""
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": p,
        "recall": r,
        "f1": f1,
    }


def save_fig(name: str, show: bool = True):
    """Lưu hình vào reports/figures/<name>.png (dùng cho báo cáo) và hiển thị."""
    ensure_dirs()
    path = os.path.join(FIG_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()
    return path


def plot_confusion(y_true, y_pred, labels=None, title="Confusion matrix", name=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    n = cm.shape[0]
    size = max(5, min(14, n * 0.45))
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
    if labels is not None and n <= 25:
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
    if name:
        save_fig(name)
    else:
        plt.show()


class ResultLogger:
    """Gom kết quả thí nghiệm thành bảng và lưu CSV để báo cáo dùng lại."""

    def __init__(self, name: str):
        self.name = name
        self.rows = []

    def add(self, **kwargs):
        self.rows.append(kwargs)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)

    def save(self):
        ensure_dirs()
        df = self.frame()
        df.to_csv(os.path.join(REPORT_DIR, f"results_{self.name}.csv"), index=False)
        return df


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.elapsed = time.time() - self.t0


def save_features(name: str, array: np.ndarray, fmt: str = "npy"):
    """Lưu đặc trưng vào features/<name>.npy (hoặc .h5)."""
    ensure_dirs()
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        np.save(path, array)
    elif fmt == "h5":
        import h5py

        with h5py.File(path, "w") as f:
            f.create_dataset("features", data=array, compression="gzip")
    else:
        raise ValueError(fmt)
    return path


def load_features(name: str, fmt: str = "npy"):
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        return np.load(path)
    import h5py

    with h5py.File(path, "r") as f:
        return f["features"][:]


def save_json(name: str, obj: dict):
    ensure_dirs()
    with open(os.path.join(REPORT_DIR, f"{name}.json"), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=float)


def download(url: str, dest: str):
    """Tải file công khai về dest (bỏ qua nếu đã có)."""
    import urllib.request

    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    print(f"Đang tải {url} ...")
    urllib.request.urlretrieve(url, dest)
    return dest

In [ ]:
%%writefile modules/tabular.py
"""Bài 1 - Dữ liệu bảng: Adult Census Income (UCI).

Nguồn công khai: https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
- ~32.5k mẫu, 14 thuộc tính (6 số + 8 phân loại), nhãn income >50K / <=50K.
- Có missing value (ký hiệu '?') ở workclass, occupation, native-country.
"""
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import LinearSVC

from common import ResultLogger, Timer, download, evaluate, save_fig

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income",
]
TARGET = "income"


# ----------------------------------------------------------------------------
# Dữ liệu
# ----------------------------------------------------------------------------
def load_adult(data_dir: str = "data") -> pd.DataFrame:
    path = download(DATA_URL, os.path.join(data_dir, "adult.data"))
    df = pd.read_csv(
        path, header=None, names=COLUMNS, skipinitialspace=True,
        na_values="?", sep=r",\s*", engine="python",
    )
    df[TARGET] = (df[TARGET].str.strip() == ">50K").astype(int)
    return df


def split_data(df: pd.DataFrame, test_size: float = 0.2, seed: int = 42):
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)


def feature_types(X: pd.DataFrame):
    num = X.select_dtypes(include="number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    return num, cat


# ----------------------------------------------------------------------------
# EDA
# ----------------------------------------------------------------------------
def run_eda(df: pd.DataFrame):
    num, cat = feature_types(df.drop(columns=[TARGET]))
    print(f"Kích thước: {df.shape} | số: {len(num)} cột | phân loại: {len(cat)} cột")
    print(f"Mẫu trùng lặp: {df.duplicated().sum()}")
    display_df = df.describe().T
    print("\n== Thống kê mô tả (số) ==")
    print(display_df.round(2))
    print("\n== Thống kê (phân loại) ==")
    print(df[cat].describe().T)

    # Missing value
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print("\n== Giá trị thiếu ==")
    print(pd.DataFrame({"missing": miss, "ratio_%": (miss / len(df) * 100).round(2)}))
    if len(miss):
        plt.figure(figsize=(6, 3))
        (miss / len(df) * 100).plot.bar(color="tab:red")
        plt.ylabel("% thiếu")
        plt.title("Tỉ lệ giá trị thiếu theo cột")
        save_fig("tab_missing")

    # Phân phối nhãn
    plt.figure(figsize=(4, 3))
    df[TARGET].value_counts().sort_index().plot.bar(color=["tab:blue", "tab:orange"])
    plt.xticks([0, 1], ["<=50K", ">50K"], rotation=0)
    plt.title("Phân phối nhãn")
    save_fig("tab_label_dist")
    print("Tỉ lệ nhãn:", df[TARGET].value_counts(normalize=True).round(3).to_dict())

    # Phân phối biến số + outlier
    fig, axes = plt.subplots(2, 3, figsize=(13, 6))
    for ax, c in zip(axes.ravel(), num):
        sns.histplot(df[c], bins=40, ax=ax, kde=False)
        ax.set_title(f"{c} (skew={df[c].skew():.2f})")
    save_fig("tab_num_hist")

    fig, axes = plt.subplots(2, 3, figsize=(13, 6))
    for ax, c in zip(axes.ravel(), num):
        sns.boxplot(x=df[TARGET], y=df[c], ax=ax)
        ax.set_title(c)
    save_fig("tab_num_box")

    # Tương quan
    plt.figure(figsize=(6, 5))
    sns.heatmap(df[num + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Ma trận tương quan")
    save_fig("tab_corr")

    # Biến phân loại vs nhãn
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, c in zip(axes.ravel(), cat):
        rate = df.groupby(c)[TARGET].mean().sort_values()
        rate.plot.barh(ax=ax)
        ax.set_title(f"P(>50K) theo {c}", fontsize=9)
        ax.tick_params(labelsize=7)
    save_fig("tab_cat_vs_target")

    # Outlier theo IQR
    rows = []
    for c in num:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        n_out = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
        rows.append({"feature": c, "outliers_IQR": n_out, "ratio_%": round(n_out / len(df) * 100, 2)})
    print("\n== Outlier (quy tắc IQR) ==")
    print(pd.DataFrame(rows))


# ----------------------------------------------------------------------------
# Pipeline cấu hình được
# ----------------------------------------------------------------------------
DEFAULT_CONFIG = {
    "num_imputer": "median",          # mean | median
    "cat_imputer": "most_frequent",   # most_frequent | constant
    "encoding": "onehot",             # onehot | ordinal
    "scaler": "standard",             # standard | minmax | none
    "feature_range": (0, 1),          # dùng cho MinMaxScaler
    "pca": None,                      # None | 0.90 | 0.95 | int
    "model": "logreg",                # logreg | linsvm | rf
    "model_params": {},
    "seed": 42,
}


def build_pipeline(X: pd.DataFrame, **overrides) -> Pipeline:
    cfg = {**DEFAULT_CONFIG, **overrides}
    num, cat = feature_types(X)

    if cfg["scaler"] == "standard":
        scaler = StandardScaler()
    elif cfg["scaler"] == "minmax":
        scaler = MinMaxScaler(feature_range=tuple(cfg["feature_range"]))
    else:
        scaler = "passthrough"
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy=cfg["num_imputer"])),
        ("scaler", scaler),
    ])

    if cfg["cat_imputer"] == "constant":
        cat_imp = SimpleImputer(strategy="constant", fill_value="Missing")
    else:
        cat_imp = SimpleImputer(strategy="most_frequent")
    if cfg["encoding"] == "onehot":
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cat_pipe = Pipeline([("imputer", cat_imp), ("encoder", enc)])

    pre = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)])
    steps = [("pre", pre)]
    if cfg["pca"] is not None:
        steps.append(("pca", PCA(n_components=cfg["pca"], random_state=cfg["seed"])))

    mp = dict(cfg["model_params"])
    if cfg["model"] == "logreg":
        model = LogisticRegression(max_iter=2000, **mp)
    elif cfg["model"] == "linsvm":
        model = LinearSVC(C=mp.pop("C", 1.0), max_iter=20000, random_state=cfg["seed"], **mp)
    elif cfg["model"] == "rf":
        model = RandomForestClassifier(
            n_estimators=mp.pop("n_estimators", 200), n_jobs=-1, random_state=cfg["seed"], **mp
        )
    else:
        raise ValueError(cfg["model"])
    steps.append(("model", model))
    return Pipeline(steps)


def run_config(X_train, X_test, y_train, y_test, **cfg) -> dict:
    """Huấn luyện + đánh giá một cấu hình; trả về dict kết quả."""
    pipe = build_pipeline(X_train, **cfg)
    with Timer() as t:
        pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    res = evaluate(y_test, pred, average="binary")
    n_feat = pipe[:-1].transform(X_test.iloc[:5]).shape[1]
    res.update({"n_features": n_feat, "fit_seconds": round(t.elapsed, 2)})
    return res, pipe


def run_grid(X_train, X_test, y_train, y_test, grid: list, name: str = "tabular") -> pd.DataFrame:
    """grid: danh sách dict cấu hình. Trả về DataFrame kết quả (và lưu CSV)."""
    log = ResultLogger(name)
    for i, cfg in enumerate(grid, 1):
        res, _ = run_config(X_train, X_test, y_train, y_test, **cfg)
        row = {k: str(v) for k, v in cfg.items() if k != "model_params"}
        row.update(res)
        log.add(**row)
        print(f"[{i}/{len(grid)}] {row['model']:<7} scaler={row.get('scaler','-'):<8} "
              f"pca={row.get('pca','None'):<5} acc={res['accuracy']:.4f} f1={res['f1']:.4f}")
    return log.save()


def main_grid(models=("logreg", "linsvm", "rf"), scalers=None, pcas=(None, 0.90, 0.95)) -> list:
    scalers = scalers or [
        {"scaler": "standard"},
        {"scaler": "minmax", "feature_range": (0, 1)},
        {"scaler": "minmax", "feature_range": (-1, 1)},
    ]
    grid = []
    for m in models:
        for s in scalers:
            for p in pcas:
                grid.append({"model": m, "pca": p, **s})
    return grid

In [ ]:
%%writefile modules/dl.py
"""Phần học sâu (điểm cộng): MLP PyTorch cho vector đặc trưng + vòng huấn luyện dùng chung."""
import copy

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from common import Timer, evaluate


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), dropout=0.3):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train, y_train, X_test, y_test, hidden=(256, 128), dropout=0.3, lr=1e-3,
              weight_decay=1e-4, epochs=60, batch_size=256, patience=8, average="macro",
              seed=42, verbose=True):
    """Huấn luyện MLP với early stopping trên 10% tập train. Trả về (metrics, model)."""
    torch.manual_seed(seed)
    dev = get_device()
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train)
    n_classes = int(max(y_train.max(), np.asarray(y_test).max()) + 1)

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=seed)
    Xt, yt = torch.from_numpy(X_tr), torch.from_numpy(y_tr).long()
    Xv, yv = torch.from_numpy(X_va).to(dev), torch.from_numpy(y_va).long().to(dev)

    model = MLP(X_tr.shape[1], n_classes, hidden, dropout).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    best, best_state, bad = float("inf"), None, 0

    with Timer() as t:
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(perm), batch_size):
                idx = perm[i:i + batch_size]
                if len(idx) < 2:
                    continue
                xb, yb = Xt[idx].to(dev), yt[idx].to(dev)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                vl = loss_fn(model(Xv), yv).item()
            if vl < best - 1e-4:
                best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    break
            if verbose and (ep + 1) % 10 == 0:
                print(f"  epoch {ep + 1:3d} val_loss={vl:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).to(dev))
        pred = logits.argmax(1).cpu().numpy()
    res = evaluate(y_test, pred, average=average)
    res["fit_seconds"] = round(t.elapsed, 2)
    res["epochs_run"] = ep + 1
    return res, model, pred

In [ ]:
import sys, warnings
sys.path.insert(0, 'modules')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_seed(42); ensure_dirs()
import tabular as T
import dl

## 1. Bài toán & dữ liệu

- **Bài toán:** dự đoán thu nhập của một người trưởng thành (`>50K` hay `<=50K`) từ thông tin nhân khẩu học — phân loại nhị phân.
- **Dữ liệu:** *Adult Census Income* (UCI Machine Learning Repository) — https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
- **Vì sao phù hợp ràng buộc của đề:** ~32.5k mẫu (đủ lớn), có **missing value** (`workclass`, `occupation`, `native-country`) và **8 cột categorical**.

> ⚠️ Đề yêu cầu các nhóm không trùng tập dữ liệu và nên thống nhất dataset với giảng viên trước. Nếu nhóm đổi dataset, chỉ cần sửa `load_adult()` trong `modules/tabular.py`.

In [ ]:
df = T.load_adult('data')
df.head()

## 2. EDA (Exploratory Data Analysis)

In [ ]:
T.run_eda(df)

**Nhận xét EDA** *(chỉnh sửa theo kết quả thực tế sau khi chạy)*:
- Nhãn **mất cân bằng** (~24% là `>50K`) ⇒ accuracy không đủ, cần theo dõi F1/recall của lớp dương.
- Missing value tập trung ở 3 cột phân loại (~2–6%) ⇒ dùng imputation theo cột (mode hoặc hằng số `Missing`).
- `capital-gain`, `capital-loss` lệch phải mạnh, nhiều giá trị 0 và outlier ⇒ ảnh hưởng đến mô hình tuyến tính/khoảng cách, cần chuẩn hoá.
- `education` và `education-num` mang cùng thông tin (tương quan cao) ⇒ dư thừa, PCA có thể giúp.
- Tỉ lệ `>50K` khác biệt rõ theo `marital-status`, `relationship`, `education`, `occupation` ⇒ các biến phân loại có sức dự báo.

## 3. Chia train/test & thiết kế pipeline

In [ ]:
X_train, X_test, y_train, y_test = T.split_data(df, test_size=0.2, seed=42)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))
num_cols, cat_cols = T.feature_types(X_train)
print('Số:', num_cols); print('Phân loại:', cat_cols)

Pipeline được xây bằng `sklearn.Pipeline` + `ColumnTransformer` và **cấu hình được từng bước** qua một dict:

| Bước | Tuỳ chọn |
|---|---|
| Imputation (số) | `mean`, `median` |
| Imputation (phân loại) | `most_frequent`, `constant` ("Missing") |
| Encoding | `onehot`, `ordinal` |
| Scaling | `standard`, `minmax` (+ `feature_range`), `none` |
| Giảm chiều | `None`, PCA giữ 90% / 95% phương sai |
| Mô hình | Logistic Regression, Linear SVM, Random Forest |

Fit toàn bộ bước tiền xử lý **chỉ trên tập train** (trong Pipeline) để tránh rò rỉ dữ liệu.

In [ ]:
cfg = dict(scaler='minmax', feature_range=(0, 1), pca=0.95, model='logreg')
pipe = T.build_pipeline(X_train, **cfg)
pipe

In [ ]:
res, pipe = T.run_config(X_train, X_test, y_train, y_test, **cfg)
print(res)

## 4. Thí nghiệm 1 — Scaler × PCA × Mô hình (27 cấu hình)

So sánh `StandardScaler`, `MinMaxScaler(0,1)`, `MinMaxScaler(-1,1)`; PCA `None / 90% / 95%`; ba mô hình.

In [ ]:
grid = T.main_grid()
print(len(grid), 'cấu hình')
df_main = T.run_grid(X_train, X_test, y_train, y_test, grid, name='tabular_main')

In [ ]:
cols = ['model', 'scaler', 'feature_range', 'pca', 'n_features', 'accuracy', 'precision', 'recall', 'f1', 'fit_seconds']
df_main[cols].sort_values('f1', ascending=False).reset_index(drop=True)

In [ ]:
d = df_main.copy()
d['scaler_cfg'] = d['scaler'] + np.where(d['scaler'] == 'minmax', d['feature_range'].fillna(''), '')
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, m in zip(axes, ['logreg', 'linsvm', 'rf']):
    pv = d[d.model == m].pivot(index='scaler_cfg', columns='pca', values='f1')
    pv = pv[['None', '0.9', '0.95']] if set(['None', '0.9', '0.95']) <= set(pv.columns) else pv
    import seaborn as sns
    sns.heatmap(pv, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, cbar=False)
    ax.set_title(f'F1 (lớp >50K) — {m}'); ax.set_xlabel('PCA'); ax.set_ylabel('')
save_fig('tab_heatmap_scaler_pca')

## 5. Thí nghiệm 2 — Imputation × Encoding

In [ ]:
grid2 = []
for m in ['logreg', 'rf']:
    for imp in ['most_frequent', 'constant']:
        for enc in ['onehot', 'ordinal']:
            grid2.append(dict(model=m, cat_imputer=imp, encoding=enc, scaler='standard', pca=None))
df_enc = T.run_grid(X_train, X_test, y_train, y_test, grid2, name='tabular_imputer_encoding')
df_enc[['model', 'cat_imputer', 'encoding', 'n_features', 'accuracy', 'f1', 'fit_seconds']]

## 6. Thí nghiệm 3 — Mất cân bằng lớp & siêu tham số

In [ ]:
grid3 = [
    dict(model='logreg', scaler='standard', model_params={}),
    dict(model='logreg', scaler='standard', model_params={'class_weight': 'balanced'}),
    dict(model='logreg', scaler='standard', model_params={'C': 0.1}),
    dict(model='logreg', scaler='standard', model_params={'C': 10}),
    dict(model='linsvm', scaler='standard', model_params={'C': 0.1}),
    dict(model='linsvm', scaler='standard', model_params={'C': 1}),
    dict(model='rf', scaler='none', model_params={}),
    dict(model='rf', scaler='none', model_params={'class_weight': 'balanced'}),
    dict(model='rf', scaler='none', model_params={'max_depth': 12, 'n_estimators': 300}),
    dict(model='rf', scaler='none', model_params={'min_samples_leaf': 5, 'n_estimators': 300}),
]
rows = []
for g in grid3:
    r, _ = T.run_config(X_train, X_test, y_train, y_test, **g)
    rows.append({'model': g['model'], 'params': str(g['model_params']), **r})
df_hp = pd.DataFrame(rows)
df_hp.to_csv('reports/results_tabular_hparams.csv', index=False)
df_hp[['model', 'params', 'accuracy', 'precision', 'recall', 'f1']]

## 7. Cấu hình tốt nhất: kiểm định chéo & phân tích lỗi

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
allres = pd.concat([df_main, df_enc], ignore_index=True)
best = df_main.sort_values('f1', ascending=False).iloc[0]
print('Cấu hình tốt nhất (thí nghiệm 1):'); print(best[['model','scaler','feature_range','pca','accuracy','f1']].to_dict())
best_cfg = dict(model=best['model'], scaler=best['scaler'],
                pca=None if best['pca'] == 'None' else float(best['pca']))
if best['scaler'] == 'minmax':
    best_cfg['feature_range'] = eval(best['feature_range'])
pipe_best = T.build_pipeline(X_train, **best_cfg)
cv = cross_val_score(pipe_best, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='f1')
print(f'5-fold CV F1 = {cv.mean():.4f} ± {cv.std():.4f}')
pipe_best.fit(X_train, y_train)
pred = pipe_best.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, pred, target_names=['<=50K', '>50K']))
plot_confusion(y_test, pred, labels=['<=50K', '>50K'], title='Confusion matrix — cấu hình tốt nhất', name='tab_confusion_best')

In [ ]:
# Độ quan trọng đặc trưng (nếu mô hình tốt nhất là Random Forest) hoặc hệ số (tuyến tính)
pre = pipe_best[:-1]
names = pre.named_steps['pre'].get_feature_names_out() if 'pca' not in pipe_best.named_steps else None
mdl = pipe_best.named_steps['model']
if names is not None:
    imp = mdl.feature_importances_ if hasattr(mdl, 'feature_importances_') else np.abs(mdl.coef_).ravel()
    top = pd.Series(imp, index=names).sort_values(ascending=False).head(15)
    plt.figure(figsize=(7, 5)); top[::-1].plot.barh(); plt.title('Top 15 đặc trưng quan trọng')
    save_fig('tab_feature_importance')
else:
    print('Mô hình tốt nhất dùng PCA nên bỏ qua feature importance trên đặc trưng gốc.')

## 8. (Điểm cộng) Pipeline học sâu — MLP trên đặc trưng đã tiền xử lý

Dùng cùng khâu tiền xử lý (impute + one-hot + standard scale) rồi huấn luyện **MLP PyTorch** (2 lớp ẩn, BatchNorm, Dropout, early stopping), so sánh với pipeline truyền thống.

In [ ]:
pre = T.build_pipeline(X_train, scaler='standard')[:-1]
Xtr_p = pre.fit_transform(X_train); Xte_p = pre.transform(X_test)
save_features('tab_preprocessed_train', Xtr_p.astype('float32')); save_features('tab_preprocessed_test', Xte_p.astype('float32'))
save_features('tab_y_train', y_train.values); save_features('tab_y_test', y_test.values)
print(Xtr_p.shape, Xte_p.shape)

dl_rows = []
for hidden, dropout in [((128, 64), 0.2), ((256, 128), 0.3), ((256, 128, 64), 0.3)]:
    r, _, _ = dl.train_mlp(Xtr_p, y_train.values, Xte_p, y_test.values, hidden=hidden, dropout=dropout, average='binary', verbose=False)
    dl_rows.append({'model': f'MLP{hidden}', 'dropout': dropout, **r})
    print(dl_rows[-1])
df_dl = pd.DataFrame(dl_rows); df_dl.to_csv('reports/results_tabular_dl.csv', index=False); df_dl

## 9. Tổng hợp & so sánh

In [ ]:
summary = pd.concat([
    df_main.sort_values('f1', ascending=False).groupby('model').head(1).assign(group='Truyền thống (tốt nhất mỗi mô hình)')[['group', 'model', 'accuracy', 'precision', 'recall', 'f1', 'fit_seconds']],
    df_dl.sort_values('f1', ascending=False).head(1).assign(group='Học sâu (MLP tốt nhất)')[['group', 'model', 'accuracy', 'precision', 'recall', 'f1', 'fit_seconds']],
], ignore_index=True)
summary.to_csv('reports/results_tabular_summary.csv', index=False)
summary

In [ ]:
ax = summary.set_index('model')[['accuracy', 'precision', 'recall', 'f1']].plot.bar(figsize=(9, 4), rot=20)
ax.set_ylim(0.5, 1); ax.set_title('So sánh các mô hình (Adult Income)'); ax.legend(loc='lower right')
save_fig('tab_summary_bar')

## 10. Kết luận *(chỉnh theo kết quả thực tế)*

- **Scaler:** *…* (ví dụ: Random Forest không phụ thuộc scaler; Logistic Regression/SVM cải thiện nhẹ khi chuẩn hoá).
- **PCA:** *…* (ví dụ: giữ 95% phương sai giảm số chiều nhưng không cải thiện, thậm chí giảm F1 do mất thông tin từ biến one-hot).
- **Imputation/Encoding:** *…*
- **Mất cân bằng lớp:** `class_weight='balanced'` tăng recall lớp `>50K` nhưng giảm precision.
- **Học sâu vs truyền thống:** *…* (dữ liệu bảng cỡ vừa thường để mô hình cây/ensemble ngang hoặc hơn MLP).
- **Hạn chế:** chưa dùng gradient boosting, chưa tinh chỉnh bằng grid/random search quy mô lớn; chỉ một lần chia train/test.

Các kết quả đều được lưu vào `reports/results_tabular_*.csv`, hình vào `reports/figures/`, đặc trưng đã tiền xử lý vào `features/`.